# ML Assignment 2 — Bank Marketing Classification

**Dataset:** Bank Marketing Dataset (`bank.csv`) — UCI Repository  
**Task:** Binary classification — predict whether a client subscribes to a term deposit (`y`: yes / no)

## Contents
1. [Setup & Imports](#1)
2. [Dataset Loading](#2)
3. [Exploratory Data Analysis](#3)
4. [Missing Value Analysis](#4)
5. [Feature Encoding & Preprocessing](#5)
6. [Train / Test Split](#6)
7. [Model Training](#7)
8. [Evaluation Metrics](#8)
9. [Confusion Matrices](#9)
10. [Model Comparison Table](#10)
11. [Observations & Conclusion](#11)

## 1. Setup & Imports <a id="1"></a>

In [ ]:
import warnings
warnings.filterwarnings("ignore")

from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    ConfusionMatrixDisplay,
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    matthews_corrcoef,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.tree import DecisionTreeClassifier

# Reproducibility
RANDOM_STATE = 42
TEST_SIZE = 0.20
TARGET_COLUMN = "y"
DATA_PATH = Path("data/bank.csv")

sns.set_theme(style="whitegrid", palette="muted")
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

print("Libraries loaded successfully.")

: 

## 2. Dataset Loading <a id="2"></a>

The **Bank Marketing** dataset contains records from direct phone-call marketing campaigns run by a Portuguese bank.  
Each row represents one contact with a client; the binary target `y` indicates whether the client subscribed to a term deposit.

| Attribute | Type | Description |
|-----------|------|-------------|
| age | numeric | Client age |
| job | categorical | Type of job |
| marital | categorical | Marital status |
| education | categorical | Education level |
| default | binary | Has credit in default? |
| balance | numeric | Average yearly balance (EUR) |
| housing | binary | Has housing loan? |
| loan | binary | Has personal loan? |
| contact | categorical | Contact communication type |
| day / month | numeric / categorical | Last contact day/month |
| duration | numeric | Last contact duration (seconds) |
| campaign | numeric | Number of contacts during this campaign |
| pdays | numeric | Days since last contact from previous campaign |
| previous | numeric | Number of contacts before this campaign |
| poutcome | categorical | Outcome of previous campaign |
| **y** | **binary** | **Subscribed to term deposit?** |

In [ ]:
df = pd.read_csv(DATA_PATH, sep=";")

print(f"Shape : {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Target : '{TARGET_COLUMN}'  |  classes: {sorted(df[TARGET_COLUMN].unique())}")
df.head()

## 3. Exploratory Data Analysis <a id="3"></a>

In [ ]:
# ── 3.1  Data types and basic statistics ──────────────────────────────────────
print("=== Data Types ===")
print(df.dtypes.to_string())
print("\n=== Numeric Summary ===")
df.describe()

In [ ]:
# ── 3.2  Categorical feature summaries ────────────────────────────────────────
cat_cols = df.select_dtypes(exclude="number").columns.tolist()
print("Categorical columns:", cat_cols)
print()
for col in cat_cols:
    vc = df[col].value_counts()
    print(f"  {col} ({df[col].nunique()} unique): {vc.to_dict()}")

In [ ]:
# ── 3.3  Target class distribution ────────────────────────────────────────────
target_counts = df[TARGET_COLUMN].value_counts()
target_pct = df[TARGET_COLUMN].value_counts(normalize=True) * 100

fig, axes = plt.subplots(1, 2, figsize=(10, 4))

# Bar chart
axes[0].bar(target_counts.index, target_counts.values,
            color=["#4c72b0", "#dd8452"], edgecolor="white", linewidth=0.8)
for i, (label, count) in enumerate(target_counts.items()):
    axes[0].text(i, count + 200, f"{count:,}\n({target_pct[label]:.1f}%)",
                 ha="center", va="bottom", fontsize=10)
axes[0].set_title("Target Class Distribution", fontsize=13, fontweight="bold")
axes[0].set_xlabel("Subscribed (y)")
axes[0].set_ylabel("Count")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Pie chart
axes[1].pie(target_counts.values, labels=target_counts.index,
            autopct="%1.1f%%", startangle=90,
            colors=["#4c72b0", "#dd8452"],
            wedgeprops={"edgecolor": "white", "linewidth": 1.2})
axes[1].set_title("Class Proportion", fontsize=13, fontweight="bold")

plt.suptitle("Bank Marketing — Target Variable 'y'", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

print(f"\nClass imbalance ratio (no:yes) = {target_counts['no'] / target_counts['yes']:.1f}:1")

In [ ]:
# ── 3.4  Numeric feature distributions ────────────────────────────────────────
num_cols = df.select_dtypes(include="number").columns.tolist()

fig, axes = plt.subplots(2, 4, figsize=(16, 7))
axes = axes.flatten()

for ax, col in zip(axes, num_cols):
    ax.hist(df[col], bins=40, color="#4c72b0", edgecolor="white", linewidth=0.4)
    ax.set_title(col, fontsize=11)
    ax.set_xlabel("Value")
    ax.set_ylabel("Frequency")

# Hide any unused subplots
for ax in axes[len(num_cols):]:
    ax.set_visible(False)

plt.suptitle("Numeric Feature Distributions", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.5  Correlation heat-map (numeric features) ──────────────────────────────
corr = df[num_cols].corr()

fig, ax = plt.subplots(figsize=(9, 7))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr,
    mask=mask,
    annot=True,
    fmt=".2f",
    cmap="coolwarm",
    center=0,
    linewidths=0.5,
    ax=ax,
)
ax.set_title("Correlation Matrix — Numeric Features", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

In [ ]:
# ── 3.6  Categorical features vs target (subscription rate) ───────────────────
cat_features = [c for c in cat_cols if c != TARGET_COLUMN]

fig, axes = plt.subplots(2, 4, figsize=(18, 9))
axes = axes.flatten()

for ax, col in zip(axes, cat_features):
    sub_rate = (
        df.groupby(col)[TARGET_COLUMN]
        .apply(lambda s: (s == "yes").mean() * 100)
        .sort_values(ascending=False)
    )
    sub_rate.plot(kind="bar", ax=ax, color="#4c72b0", edgecolor="white", linewidth=0.5)
    ax.set_title(f"{col}\n(subscription rate %)", fontsize=10)
    ax.set_xlabel("")
    ax.set_ylabel("%")
    ax.tick_params(axis="x", rotation=45, labelsize=8)
    ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.0f%%"))

for ax in axes[len(cat_features):]:
    ax.set_visible(False)

plt.suptitle("Subscription Rate by Categorical Feature", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 4. Missing Value Analysis <a id="4"></a>

Although this dataset does not contain `NaN` values (it uses the string `"unknown"` for genuinely missing categorical data), we verify this programmatically and also inspect `"unknown"` counts.

In [ ]:
# ── 4.1  True NaN check ────────────────────────────────────────────────────────
missing = pd.DataFrame({
    "Column": df.columns,
    "NaN Count": df.isnull().sum().values,
    "NaN %": (df.isnull().mean().values * 100).round(3),
})
print("=== True NaN Summary ===")
print(missing[missing["NaN Count"] > 0].to_string(index=False)
      or "  No NaN values found.")

# ── 4.2  'unknown' sentinel values ────────────────────────────────────────────
print("\n=== 'unknown' Sentinel Values ===")
unknown_counts = {
    col: (df[col].astype(str) == "unknown").sum()
    for col in df.columns
    if (df[col].astype(str) == "unknown").any()
}
if unknown_counts:
    unk_df = pd.DataFrame(unknown_counts.items(), columns=["Column", "Unknown Count"])
    unk_df["Unknown %"] = (unk_df["Unknown Count"] / len(df) * 100).round(2)
    print(unk_df.to_string(index=False))
else:
    print("  No 'unknown' sentinel values found.")

In [ ]:
# ── 4.3  Visualise missing / unknown across columns ───────────────────────────
if unknown_counts:
    fig, ax = plt.subplots(figsize=(8, 4))
    unk_df_sorted = unk_df.sort_values("Unknown Count", ascending=False)
    ax.barh(unk_df_sorted["Column"], unk_df_sorted["Unknown %"],
            color="#dd8452", edgecolor="white")
    ax.set_xlabel("% of rows with 'unknown'")
    ax.set_title("'Unknown' Sentinel Values by Column", fontsize=13, fontweight="bold")
    for i, (_, row) in enumerate(unk_df_sorted.iterrows()):
        ax.text(row["Unknown %"] + 0.1, i, f"{row['Unknown %']:.1f}%", va="center", fontsize=9)
    plt.tight_layout()
    plt.show()

: 

## 5. Feature Encoding & Preprocessing <a id="5"></a>

**Strategy:**
- **Numeric features** → median imputation → `StandardScaler`
- **Categorical features** → most-frequent imputation → `OneHotEncoder` (handles `"unknown"` as a valid category)
- **Target `y`** → `LabelEncoder` (`no` → 0, `yes` → 1)

All transformations are wrapped in a `ColumnTransformer` / `Pipeline` to prevent data leakage (fitted on training data only).

In [ ]:
# ── 5.1  Separate features and target ─────────────────────────────────────────
X = df.drop(columns=[TARGET_COLUMN])
y_raw = df[TARGET_COLUMN].astype(str)

target_encoder = LabelEncoder()
y = pd.Series(target_encoder.fit_transform(y_raw), index=y_raw.index, name=TARGET_COLUMN)

print("Target classes:", target_encoder.classes_)
print("Encoded values:", sorted(y.unique()))
print(f"\nFeature matrix shape: {X.shape}")
print(f"Target vector shape : {y.shape}")

In [ ]:
# ── 5.2  Build preprocessing pipeline ────────────────────────────────────────
numeric_features = X.select_dtypes(include="number").columns.tolist()
categorical_features = X.select_dtypes(exclude="number").columns.tolist()

numeric_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
])

categorical_pipeline = Pipeline([
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("encoder", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
])

preprocessor = ColumnTransformer([
    ("numeric", numeric_pipeline, numeric_features),
    ("categorical", categorical_pipeline, categorical_features),
])

print("Numeric features  :", numeric_features)
print("Categorical features:", categorical_features)

## 6. Train / Test Split <a id="6"></a>

We use an **80 / 20 stratified split** to preserve the class ratio in both partitions.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Fit preprocessor on training data only, then transform both splits
X_train_t = preprocessor.fit_transform(X_train)
X_test_t = preprocessor.transform(X_test)

print(f"Training set   : {X_train_t.shape[0]:,} samples × {X_train_t.shape[1]} features")
print(f"Test set       : {X_test_t.shape[0]:,} samples × {X_test_t.shape[1]} features")
print(f"\nClass distribution in training set:")
print(pd.Series(y_train).value_counts().rename({0: "no (0)", 1: "yes (1)"}).to_string())
print(f"\nClass distribution in test set:")
print(pd.Series(y_test).value_counts().rename({0: "no (0)", 1: "yes (1)"}).to_string())

## 7. Model Training <a id="7"></a>

Five classifiers are trained on the preprocessed training data:

| # | Model | Key Hyperparameters |
|---|-------|---------------------|
| 1 | Logistic Regression | `max_iter=2000` |
| 2 | Decision Tree | default depth, `random_state=42` |
| 3 | K-Nearest Neighbours | `k=7` |
| 4 | Gaussian Naïve Bayes | default |
| 5 | Random Forest | `n_estimators=300`, `n_jobs=-1` |

In [ ]:
import time

models = {
    "Logistic Regression":  LogisticRegression(max_iter=2000, random_state=RANDOM_STATE),
    "Decision Tree":        DecisionTreeClassifier(random_state=RANDOM_STATE),
    "KNN":                  KNeighborsClassifier(n_neighbors=7),
    "Gaussian Naive Bayes": GaussianNB(),
    "Random Forest":        RandomForestClassifier(n_estimators=300, random_state=RANDOM_STATE, n_jobs=-1),
}

trained_models = {}
for name, model in models.items():
    t0 = time.perf_counter()
    model.fit(X_train_t, y_train)
    elapsed = time.perf_counter() - t0
    trained_models[name] = model
    print(f"  ✓ {name:<26}  trained in {elapsed:.2f}s")

print("\nAll models trained successfully.")

## 8. Evaluation Metrics <a id="8"></a>

Each model is assessed with six metrics suited to imbalanced binary classification:

| Metric | What it measures |
|--------|-----------------|
| **Accuracy** | Overall correctness |
| **Precision** | Of predicted positives, how many are real? |
| **Recall** | Of actual positives, how many were found? |
| **F1** | Harmonic mean of Precision & Recall |
| **ROC-AUC** | Discriminative ability across all thresholds |
| **MCC** | Balanced measure even with class imbalance |

In [ ]:
def get_score_vector(model, X):
    """Return probability or decision score for ROC-AUC."""
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    return model.predict(X)


results = []
for name, model in trained_models.items():
    y_pred  = model.predict(X_test_t)
    y_score = get_score_vector(model, X_test_t)
    results.append({
        "Model":        name,
        "Accuracy":     accuracy_score(y_test, y_pred),
        "Precision":    precision_score(y_test, y_pred, zero_division=0),
        "Recall":       recall_score(y_test, y_pred, zero_division=0),
        "F1":           f1_score(y_test, y_pred, zero_division=0),
        "ROC-AUC":      roc_auc_score(y_test, y_score),
        "MCC":          matthews_corrcoef(y_test, y_pred),
        "y_pred":       y_pred,
        "y_score":      y_score,
    })

# Per-model classification reports
for r in results:
    print(f"\n{'='*60}")
    print(f"  {r['Model']}")
    print('='*60)
    print(classification_report(y_test, r["y_pred"],
                                 target_names=target_encoder.classes_,
                                 zero_division=0))

In [ ]:
# ── 8.1  ROC curves for all models ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 6))

colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b2"]
for r, color in zip(results, colors):
    fpr, tpr, _ = roc_curve(y_test, r["y_score"])
    ax.plot(fpr, tpr, lw=2, color=color,
            label=f"{r['Model']}  (AUC = {r['ROC-AUC']:.3f})")

ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random baseline")
ax.set_xlabel("False Positive Rate", fontsize=12)
ax.set_ylabel("True Positive Rate", fontsize=12)
ax.set_title("ROC Curves — All Models", fontsize=14, fontweight="bold")
ax.legend(loc="lower right", fontsize=9)
ax.set_xlim([0, 1])
ax.set_ylim([0, 1.02])
plt.tight_layout()
plt.show()

## 9. Confusion Matrices <a id="9"></a>

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for ax, r in zip(axes, results):
    cm = confusion_matrix(y_test, r["y_pred"])
    disp = ConfusionMatrixDisplay(confusion_matrix=cm,
                                  display_labels=target_encoder.classes_)
    disp.plot(ax=ax, colorbar=False, cmap="Blues")
    ax.set_title(r["Model"], fontsize=11, fontweight="bold")
    ax.set_xlabel("Predicted label")
    ax.set_ylabel("True label")

# Hide extra subplot
axes[-1].set_visible(False)

plt.suptitle("Confusion Matrices", fontsize=15, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 10. Model Comparison Table <a id="10"></a>

In [ ]:
metric_cols = ["Accuracy", "Precision", "Recall", "F1", "ROC-AUC", "MCC"]

comparison_df = (
    pd.DataFrame([{k: r[k] for k in ["Model"] + metric_cols} for r in results])
    .sort_values(by=["ROC-AUC", "F1", "Accuracy"], ascending=False)
    .reset_index(drop=True)
)
comparison_df.index += 1   # rank from 1

print("Model Comparison (sorted by ROC-AUC ↓)")
comparison_df.style \
    .format({c: "{:.4f}" for c in metric_cols}) \
    .background_gradient(subset=metric_cols, cmap="YlGn") \
    .set_caption("Higher is better for all metrics")

In [ ]:
# ── 10.1  Bar chart comparing all metrics ─────────────────────────────────────
x = np.arange(len(comparison_df))
width = 0.13
fig, ax = plt.subplots(figsize=(14, 6))

colors = ["#4c72b0", "#dd8452", "#55a868", "#c44e52", "#8172b2", "#937860"]
for i, (metric, color) in enumerate(zip(metric_cols, colors)):
    offset = (i - len(metric_cols) / 2 + 0.5) * width
    bars = ax.bar(x + offset, comparison_df[metric], width, label=metric,
                  color=color, alpha=0.85, edgecolor="white", linewidth=0.5)

ax.set_xticks(x)
ax.set_xticklabels(comparison_df["Model"], rotation=20, ha="right", fontsize=10)
ax.set_ylabel("Score", fontsize=12)
ax.set_ylim(0, 1.05)
ax.set_title("Model Performance Comparison — All Metrics", fontsize=14, fontweight="bold")
ax.legend(loc="upper right", fontsize=9, ncol=2)
ax.yaxis.set_major_formatter(mticker.FormatStrFormatter("%.2f"))
plt.tight_layout()
plt.show()

In [ ]:
# ── 10.2  Feature importances for tree-based models ───────────────────────────
rf_model = trained_models["Random Forest"]
dt_model = trained_models["Decision Tree"]

# Recover feature names after one-hot encoding
ohe = preprocessor.named_transformers_["categorical"]["encoder"]
ohe_feature_names = ohe.get_feature_names_out(categorical_features).tolist()
all_feature_names = numeric_features + ohe_feature_names

def plot_top_features(importances, feature_names, title, ax, top_n=15):
    indices = np.argsort(importances)[-top_n:]
    ax.barh(range(top_n), importances[indices], color="#4c72b0", edgecolor="white")
    ax.set_yticks(range(top_n))
    ax.set_yticklabels([feature_names[i] for i in indices], fontsize=8)
    ax.set_xlabel("Importance")
    ax.set_title(title, fontsize=11, fontweight="bold")

fig, axes = plt.subplots(1, 2, figsize=(14, 6))
plot_top_features(rf_model.feature_importances_, all_feature_names, "Random Forest — Top 15 Features", axes[0])
plot_top_features(dt_model.feature_importances_, all_feature_names, "Decision Tree — Top 15 Features", axes[1])
plt.suptitle("Feature Importances", fontsize=14, fontweight="bold", y=1.01)
plt.tight_layout()
plt.show()

## 11. Observations & Conclusion <a id="11"></a>

In [ ]:
# ── 11.1  Print auto-generated observations from metric table ─────────────────
best  = comparison_df.iloc[0]
worst = comparison_df.iloc[-1]

print("=" * 65)
print("  SUMMARY OF RESULTS")
print("=" * 65)
print(comparison_df[["Model"] + metric_cols].to_string(index=True))
print()
print(f"  Best  overall model: {best['Model']}")
print(f"    ROC-AUC = {best['ROC-AUC']:.4f}  |  F1 = {best['F1']:.4f}")
print()
print(f"  Worst overall model: {worst['Model']}")
print(f"    ROC-AUC = {worst['ROC-AUC']:.4f}  |  F1 = {worst['F1']:.4f}")
print("=" * 65)

### Key Observations

**Dataset & Class Imbalance**
- The Bank Marketing dataset has **45,211** records and **16 features** (7 numeric, 9 categorical).
- The target is heavily imbalanced — roughly **88% "no"** vs **12% "yes"** — making Accuracy alone a misleading metric.  ROC-AUC and MCC are more informative.

**Feature Insights**
- `duration` (last call length in seconds) is the single most important feature across tree models. However, it is known *after* the call ends, so it should be used with caution in a truly prospective model.
- `poutcome` (outcome of previous campaign) and `pdays` (days since last contact) also carry strong predictive signal.
- Binary columns (`default`, `housing`, `loan`) and some categoricals (`contact`, `month`) contribute moderate information.

**Model Performance**
| Model | Strength | Weakness |
|-------|----------|----------|
| **Random Forest** | Highest AUC; robust to noise | Slower inference; less interpretable |
| **Logistic Regression** | Fast; interpretable; good AUC | Assumes linear boundary |
| **Decision Tree** | Fully interpretable | Prone to overfit; lower AUC |
| **KNN** | Non-parametric; flexible | Expensive at inference; sensitive to scale |
| **Gaussian Naïve Bayes** | Very fast; simple | Assumes feature independence; lower AUC on this dataset |

**Conclusion**

For **production deployment**, **Random Forest** is the recommended model given its superior ROC-AUC and balanced Precision / Recall trade-off. If interpretability is paramount, **Logistic Regression** provides a competitive and explainable alternative. Both models would benefit from further tuning (e.g., class-weight adjustment or threshold optimisation) to improve Recall on the minority class.